In [ ]:
import pandas as pd
import numpy as np

# Load the dataset
input_path = '/Users/marcelosilva/Desktop/early-obesity-prediction/C-Feature Engeneering/2FE10-20.csv'
df = pd.read_csv(input_path)

print("Dataset loaded successfully!")
print(f"Shape: {df.shape}")
print("\n" + "="*70)

# List of the next 10 variables for analysis (prenatal and breastfeeding)
prenatal_breastfeeding_vars = [
    'k03_prenatal',
    'k04_prenatal_semanas', 
    'k05_prenatal_consultas',
    'k06_peso_engravidar',
    'k07_peso_final',
    'k08_quilos',
    'k11_amamentou',
    'k12_tempo',
    'k13_tempo_medida',
    'k15_recebeu'
]

def analyze_variable_comprehensive(df, var_name, max_unique=20):
    """
    Comprehensive analysis of a specific variable
    
    Args:
        df: DataFrame
        var_name: variable name
        max_unique: maximum number of unique values to display
    """
    print(f"\n[ANALYSIS] DETAILED ANALYSIS: {var_name}")
    print("-" * 60)
    
    # Check if the variable exists
    if var_name not in df.columns:
        print(f"[FAIL] VARIABLE NOT FOUND IN DATASET")
        return
    
    # Basic information
    total_rows = len(df)
    unique_count = df[var_name].nunique()
    missing_count = df[var_name].isna().sum()
    missing_pct = (missing_count / total_rows) * 100
    
    print(f"Data type: {df[var_name].dtype}")
    print(f"Total records: {total_rows:,}")
    print(f"Unique values: {unique_count}")
    print(f"Missing values: {missing_count} ({missing_pct:.2f}%)")
    print(f"Valid values: {total_rows - missing_count:,} ({100-missing_pct:.2f}%)")
    
    # Type-specific analysis
    if df[var_name].dtype in ['int64', 'float64']:
        # Numeric variable
        valid_data = df[var_name].dropna()
        if len(valid_data) > 0:
            print(f"\n📈 DESCRIPTIVE STATISTICS:")
            stats = valid_data.describe()
            print(f"Mean: {stats['mean']:.2f}")
            print(f"Median: {stats['50%']:.2f}")
            print(f"Standard deviation: {stats['std']:.2f}")
            print(f"Minimum: {stats['min']}")
            print(f"Maximum: {stats['max']}")
            print(f"Q1: {stats['25%']}")
            print(f"Q3: {stats['75%']}")
            
            # Identify outliers (IQR method)
            Q1 = stats['25%']
            Q3 = stats['75%']
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            outliers = valid_data[(valid_data < lower_bound) | (valid_data > upper_bound)]
            outlier_pct = (len(outliers) / len(valid_data)) * 100
            
            if len(outliers) > 0:
                print(f"Outliers (IQR method): {len(outliers)} ({outlier_pct:.1f}%)")
                print(f"Outlier range: {outliers.min():.2f} - {outliers.max():.2f}")
        
        # If there are few unique values, show distribution
        if unique_count <= max_unique:
            print(f"\n[INFO] VALUE DISTRIBUTION:")
            value_counts = df[var_name].value_counts().sort_index()
            for value, count in value_counts.items():
                percentage = (count / total_rows) * 100
                print(f"  {value}: {count:,} ({percentage:.2f}%)")
    
    else:
        # Categorical variable
        print(f"\n[INFO] VALUE DISTRIBUTION:")
        if unique_count <= max_unique:
            # Show all categories
            value_counts = df[var_name].value_counts(dropna=False)
            for value, count in value_counts.items():
                percentage = (count / total_rows) * 100
                if pd.isna(value):
                    print(f"  [MISSING]: {count:,} ({percentage:.2f}%)")
                else:
                    print(f"  '{value}': {count:,} ({percentage:.2f}%)")
        else:
            # Show only the most frequent
            print(f"Top {max_unique} most frequent values:")
            value_counts = df[var_name].value_counts().head(max_unique)
            for value, count in value_counts.items():
                percentage = (count / total_rows) * 100
                print(f"  '{value}': {count:,} ({percentage:.2f}%)")
            
            if missing_count > 0:
                print(f"  [MISSING]: {missing_count:,} ({missing_pct:.2f}%)")
    
    # Identify possible issues and suggestions
    print(f"\n[SEARCH] OBSERVATIONS AND SUGGESTIONS:")
    
    # Missing values
    if missing_count > 0:
        if missing_pct > 20:
            print(f"  [WARNING] HIGH missing percentage ({missing_pct:.1f}%) - consider imputation or removal")
        elif missing_pct > 10:
            print(f"  [WARNING] Moderate missing percentage ({missing_pct:.1f}%) - apply imputation")
        elif missing_pct > 1:
            print(f"  [WARNING] Some missing values ({missing_pct:.1f}%) - median imputation recommended")
        else:
            print(f"  ℹ️ Few missing values ({missing_pct:.1f}%)")
    else:
        print(f"  ✅ No missing values")
    
    # Variability analysis
    if unique_count == 1:
        print(f"  [WARNING] CONSTANT variable - consider removal")
    elif unique_count == 2:
        print(f"  ℹ️ Binary variable - candidate for 0/1 encoding")
    elif df[var_name].dtype == 'object' and unique_count < 10:
        print(f"  ℹ️ Categorical variable ({unique_count} categories) - candidate for dummies")
    
    # For categorical variables, identify rare categories
    if df[var_name].dtype == 'object' and unique_count > 2:
        value_counts = df[var_name].value_counts()
        rare_categories = value_counts[value_counts / total_rows < 0.01]  # <1%
        if len(rare_categories) > 0:
            print(f"  [WARNING] {len(rare_categories)} category(ies) with <1%:")
            for cat, count in rare_categories.items():
                pct = (count / total_rows) * 100
                print(f"      '{cat}': {count} ({pct:.2f}%) - consider grouping")
    
    # For numeric variables, suggest transformations
    if df[var_name].dtype in ['int64', 'float64'] and len(df[var_name].dropna()) > 0:
        valid_data = df[var_name].dropna()
        if len(outliers) > 0 and outlier_pct > 5:
            print(f"  [WARNING] Many outliers ({outlier_pct:.1f}%) - consider transformation or capping")
        
        # Check distribution (approximate normality)
        skewness = valid_data.skew()
        if abs(skewness) > 2:
            print(f"  ℹ️ Skewed distribution (skew={skewness:.2f}) - consider log transform")
        elif abs(skewness) > 1:
            print(f"  ℹ️ Slight skewness (skew={skewness:.2f})")

# Analyze each variable
print("ANALYSIS OF PRENATAL AND BREASTFEEDING VARIABLES")
print("=" * 70)

for i, var in enumerate(prenatal_breastfeeding_vars, 1):
    print(f"\n[{i}/10] ", end="")
    analyze_variable_comprehensive(df, var)

print("\n" + "=" * 70)
print("ANALYSIS COMPLETED")
print("=" * 70)
print("\n[INFO] SUMMARY FOR FEATURE ENGINEERING:")
print("1. Identify variables for median imputation")
print("2. Evaluate the need for transformations on numeric variables")
print("3. Plan binary encoding for simple categorical variables")
print("4. Consider grouping rare categories")
print("5. Detect outliers that need treatment")
print("6. Evaluate creation of derived variables (e.g., gestational weight gain)")

In [8]:
import pandas as pd
import numpy as np
import os

def feature_engineering_prenatal_breastfeeding():
    """
    Feature Engineering das variables pré-natais e de breastfeeding
    Abordagem limpa: uma variable por vez, sem caps artificiais
    """
    
    # Load dataset
    input_path = '/Users/marcelosilva/Desktop/early-obesity-prediction/C-Feature Engeneering/2FE10-20.csv'
    df = pd.read_csv(input_path)
    
    print(f"Dataset carregado: {df.shape}")
    print("Processando variables pré-natais e breastfeeding...")
    print("=" * 60)
    
    # Work with a copy
    df_fe = df.copy()
    
    # 1. k03_prenatal - Binary with NaN
    if 'k03_prenatal' in df_fe.columns:
        k03_original = df_fe['k03_prenatal'].copy()
        k03_processed = k03_original.copy()
        
        # Convert to binary: 1=Sim, 0=Não, NaN=missing/"não sabe"
        k03_processed = k03_processed.map({
            'Sim': 1,
            'Não': 0,
            'Não sabe/Não quis responder': np.nan
        })
        
        df_fe['k03_prenatal'] = k03_processed
        
        print(f"[OK] k03_prenatal processado:")
        print(f"   Sim (1): {(k03_processed == 1).sum()}")
        print(f"   Não (0): {(k03_processed == 0).sum()}")
        print(f"   Missing/NS (NaN): {k03_processed.isna().sum()}")
    
    # 2. k04_prenatal_semanas - 99→median, missing→NaN
    if 'k04_prenatal_semanas' in df_fe.columns:
        k04_original = df_fe['k04_prenatal_semanas'].copy()
        
        # Calculate median from valid prenatal cases (excluded 99 and missing)
        valid_prenatal = (df['k03_prenatal'] == 'Sim') & (k04_original < 99) & k04_original.notna()
        
        if valid_prenatal.sum() > 0:
            median_weeks = k04_original[valid_prenatal].median()
            k04_processed = k04_original.replace(99, median_weeks)
        else:
            k04_processed = k04_original.copy()
        
        df_fe['k04_prenatal_semanas'] = k04_processed
        
        print(f"[OK] k04_prenatal_semanas processado:")
        print(f"   Valuees 99 → mediana ({median_weeks if valid_prenatal.sum() > 0 else 'N/A'})")
        print(f"   Missing mantidos: {k04_processed.isna().sum()}")
        print(f"   Valuees válidos: {k04_processed.notna().sum()}")
    
    # 3. k05_prenatal_consultas - 99→median, missing→0
    if 'k05_prenatal_consultas' in df_fe.columns:
        k05_original = df_fe['k05_prenatal_consultas'].copy()
        
        # Calculate median from valid prenatal cases
        valid_prenatal = (df['k03_prenatal'] == 'Sim') & (k05_original < 99) & k05_original.notna()
        
        if valid_prenatal.sum() > 0:
            median_consult = k05_original[valid_prenatal].median()
            k05_processed = k05_original.replace(99, median_consult)
        else:
            k05_processed = k05_original.copy()
        
        # Missing → 0 (no prenatal = 0 consultations)
        k05_processed = k05_processed.fillna(0)
        df_fe['k05_prenatal_consultas'] = k05_processed
        
        print(f"[OK] k05_prenatal_consultas processado:")
        print(f"   Valuees 99 → mediana ({median_consult if valid_prenatal.sum() > 0 else 'N/A'})")
        print(f"   Missing → 0: {k05_original.isna().sum()} cases")
        print(f"   Valuees finais válidos: {k05_processed.notna().sum()}")
    
    # 4. k06_peso_engravidar - 999.9→median, missing→NaN
    if 'k06_peso_engravidar' in df_fe.columns:
        k06_original = df_fe['k06_peso_engravidar'].copy()
        
        # Replace 999.9 with median
        valid_weights = k06_original[(k06_original != 999.9) & k06_original.notna()]
        if len(valid_weights) > 0:
            median_weight = valid_weights.median()
            k06_processed = k06_original.replace(999.9, median_weight)
        else:
            k06_processed = k06_original.copy()
        
        df_fe['k06_peso_engravidar'] = k06_processed
        
        missing_count = k06_processed.isna().sum()
        code_999_count = (k06_original == 999.9).sum()
        
        print(f"[OK] k06_peso_engravidar processado:")
        print(f"   Códigos 999.9 → mediana ({median_weight if len(valid_weights) > 0 else 'N/A'}kg): {code_999_count}")
        print(f"   Missing mantidos: {missing_count}")
        print(f"   Valuees válidos: {k06_processed.notna().sum()}")
    
    # 5. k07_peso_final - 999.9→median, missing→NaN
    if 'k07_peso_final' in df_fe.columns:
        k07_original = df_fe['k07_peso_final'].copy()
        
        # Replace 999.9 with median
        valid_weights = k07_original[(k07_original != 999.9) & k07_original.notna()]
        if len(valid_weights) > 0:
            median_weight = valid_weights.median()
            k07_processed = k07_original.replace(999.9, median_weight)
        else:
            k07_processed = k07_original.copy()
        
        df_fe['k07_peso_final'] = k07_processed
        
        missing_count = k07_processed.isna().sum()
        code_999_count = (k07_original == 999.9).sum()
        
        print(f"[OK] k07_peso_final processado:")
        print(f"   Códigos 999.9 → mediana ({median_weight if len(valid_weights) > 0 else 'N/A'}kg): {code_999_count}")
        print(f"   Missing mantidos: {missing_count}")
        print(f"   Valuees válidos: {k07_processed.notna().sum()}")
    
    # 6. k08_quilos - 99.9→median, missing→NaN, extreme values→NaN
    if 'k08_quilos' in df_fe.columns:
        k08_original = df_fe['k08_quilos'].copy()
        
        # Replace 99.9 with median
        valid_gains = k08_original[(k08_original != 99.9) & k08_original.notna()]
        if len(valid_gains) > 0:
            median_gain = valid_gains.median()
            k08_processed = k08_original.replace(99.9, median_gain)
        else:
            k08_processed = k08_original.copy()
        
        # Convert extreme negative values to NaN (likely bad data)
        # Small negative values might be real (weight loss), extreme ones are likely errors
        k08_processed = k08_processed.where(k08_processed > -5, np.nan)
        
        df_fe['k08_quilos'] = k08_processed
        
        missing_count = k08_processed.isna().sum()
        code_999_count = (k08_original == 99.9).sum()
        extreme_negative_count = (k08_original <= -5).sum() if k08_original.notna().sum() > 0 else 0
        
        print(f"[OK] k08_quilos processado:")
        print(f"   Códigos 99.9 → mediana ({median_gain if len(valid_gains) > 0 else 'N/A'}kg): {code_999_count}")
        print(f"   Valuees extremamente negativos (≤-5kg) → NaN: {extreme_negative_count}")
        print(f"   Missing mantidos: {missing_count}")
        print(f"   Valuees válidos: {k08_processed.notna().sum()}")
    
    # 7. k11_amamentou - Binary with NaN
    if 'k11_amamentou' in df_fe.columns:
        k11_original = df_fe['k11_amamentou'].copy()
        
        # Convert to binary: 1=Sim, 0=Não, NaN=missing/"não sabe"
        k11_processed = k11_original.map({
            'Sim': 1,
            'Não': 0,
            'Não sabe/Não quis responder': np.nan
        })
        
        df_fe['k11_amamentou'] = k11_processed
        
        print(f"[OK] k11_amamentou processado:")
        print(f"   Sim (1): {(k11_processed == 1).sum()}")
        print(f"   Não (0): {(k11_processed == 0).sum()}")
        print(f"   Missing/NS (NaN): {k11_processed.isna().sum()}")
    
    # 8. k12_tempo + k13_tempo_medida → tempo_primeira_mamada_horas
    if 'k12_tempo' in df_fe.columns and 'k13_tempo_medida' in df_fe.columns:
        k12_original = df_fe['k12_tempo'].copy()
        k13_original = df_fe['k13_tempo_medida'].copy()
        
        # Get position of k12_tempo to insert new variable there
        k12_position = df_fe.columns.get_loc('k12_tempo')
        
        # Convert to hours
        tempo_horas = k12_original.copy()
        
        # Where k13 is "Dias", multiply k12 by 24
        dias_mask = (k13_original == 'Dias') & k12_original.notna()
        tempo_horas.loc[dias_mask] = k12_original.loc[dias_mask] * 24
        
        # Remove originals first
        df_fe = df_fe.drop(columns=['k12_tempo', 'k13_tempo_medida'])
        
        # Insert new variable at the original k12 position
        df_fe.insert(k12_position, 'tempo_primeira_mamada_horas', tempo_horas)
        
        dias_conversions = dias_mask.sum()
        missing_count = tempo_horas.isna().sum()
        
        print(f"[OK] k12_tempo + k13_tempo_medida → tempo_primeira_mamada_horas:")
        print(f"   Conversões Dias→Horas: {dias_conversions}")
        print(f"   Missing mantidos: {missing_count}")
        print(f"   Valuees válidos: {tempo_horas.notna().sum()}")
    
    # 9. k15_recebeu - Binary with NaN
    if 'k15_recebeu' in df_fe.columns:
        k15_original = df_fe['k15_recebeu'].copy()
        
        # Convert to binary: 1=Sim, 0=Não, NaN=missing/"não sabe"
        k15_processed = k15_original.map({
            'Sim': 1,
            'Não': 0,
            'Não sabe/Não quis responder': np.nan
        })
        
        df_fe['k15_recebeu'] = k15_processed
        
        print(f"[OK] k15_recebeu processado:")
        print(f"   Sim (1): {(k15_processed == 1).sum()}")
        print(f"   Não (0): {(k15_processed == 0).sum()}")
        print(f"   Missing/NS (NaN): {k15_processed.isna().sum()}")
    
    # Save result
    output_dir = '/Users/marcelosilva/Desktop/early-obesity-prediction/C-Feature Engeneering'
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, '3FE20-30.csv')
    df_fe.to_csv(output_path, index=False)
    
    print("\n" + "=" * 60)
    print("PROCESSAMENTO CONCLUÍDO!")
    print("=" * 60)
    print(f"Original dataset: {df.shape}")
    print(f"Dataset processado: {df_fe.shape}")
    print(f"Arquivo salvo: {output_path}")
    
    # Variables summary
    original_vars = ['k03_prenatal', 'k04_prenatal_semanas', 'k05_prenatal_consultas',
                    'k06_peso_engravidar', 'k07_peso_final', 'k08_quilos',
                    'k11_amamentou', 'k12_tempo', 'k13_tempo_medida', 'k15_recebeu']
    
    processed_vars = ['k03_prenatal', 'k04_prenatal_semanas', 'k05_prenatal_consultas',
                     'k06_peso_engravidar', 'k07_peso_final', 'k08_quilos',
                     'k11_amamentou', 'tempo_primeira_mamada_horas', 'k15_recebeu']
    
    vars_present_original = [v for v in original_vars if v in df.columns]
    vars_present_processed = [v for v in processed_vars if v in df_fe.columns]
    
    print(f"Variáveis processadas: {len(vars_present_original)} → {len(vars_present_processed)}")
    
    # Missing values check
    missing_summary = df_fe[vars_present_processed].isnull().sum()
    vars_with_missing = missing_summary[missing_summary > 0]
    
    print(f"\nVariáveis com missing values:")
    if len(vars_with_missing) > 0:
        for var, count in vars_with_missing.items():
            pct = (count / len(df_fe)) * 100
            print(f"  {var}: {count} ({pct:.1f}%)")
    else:
        print("  Nenhuma variable com missing values")
    
    print(f"\nPrimeiras 10 colunas do dataset processado:")
    for i, col in enumerate(df_fe.columns[:10], 1):
        print(f"  {i}. {col}")
    
    return df_fe

# Execute
if __name__ == "__main__":
    result = feature_engineering_prenatal_breastfeeding()

Dataset carregado: (8236, 60)
Processando variables pré-natais e breastfeeding...
[OK] k03_prenatal processado:
   Sim (1): 6010
   Não (0): 105
   Missing/NS (NaN): 2121
[OK] k04_prenatal_semanas processado:
   Valuees 99 → mediana (6.0)
   Missing mantidos: 2226
   Valuees válidos: 6010
[OK] k05_prenatal_consultas processado:
   Valuees 99 → mediana (9.0)
   Missing → 0: 2226 cases
   Valuees finais válidos: 8236
[OK] k06_peso_engravidar processado:
   Códigos 999.9 → mediana (60.0kg): 614
   Missing mantidos: 2108
   Valuees válidos: 6128
[OK] k07_peso_final processado:
   Códigos 999.9 → mediana (72.0kg): 725
   Missing mantidos: 2106
   Valuees válidos: 6130
[OK] k08_quilos processado:
   Códigos 99.9 → mediana (11.0kg): 716
   Valuees extremamente negativos (≤-5kg) → NaN: 0
   Missing mantidos: 2104
   Valuees válidos: 6132
[OK] k11_amamentou processado:
   Sim (1): 5857
   Não (0): 266
   Missing/NS (NaN): 2113
[OK] k12_tempo + k13_tempo_medida → tempo_primeira_mamada_horas:
   

# Feature Engineering - Prenatal and Breastfeeding Variables

## Processing Summary

This notebook executed **Feature Engineering** for variables related to **prenatal care and breastfeeding** (k03 to k15), transforming categorical data into numerical format and strategically handling missing values.

## Transformations Applied

### 1. Binary Variables (Yes/No)
- **k03_prenatal**: Had prenatal care → 1=Yes, 0=No, NaN=Don't know
- **k11_amamentou**: Breastfed → 1=Yes, 0=No, NaN=Don't know  
- **k15_recebeu**: Received guidance → 1=Yes, 0=No, NaN=Don't know

### 2. Numerical Variables with Special Codes
- **k04_prenatal_semanas**: Prenatal weeks → 99 replaced with median
- **k05_prenatal_consultas**: Number of consultations → 99 replaced with median, missing→0
- **k06_peso_engravidar**: Weight at pregnancy → 999.9 replaced with median
- **k07_peso_final**: Final pregnancy weight → 999.9 replaced with median
- **k08_quilos**: Weight gain → 99.9 replaced with median, values ≤-5kg→NaN

### 3. Derived Variable
- **tempo_primeira_mamada_horas**: Combination of k12_tempo + k13_tempo_medida
    - Converts days to hours (multiplies by 24)
    - Replaces the two original variables with a single one in hours

## Processing Results

- **Original dataset**: 8,236 records × 60 columns
- **Processed dataset**: 8,236 records × 59 columns  
- **Generated file**: `3FE20-30.csv`

### Variables with Remaining Missing Values
- **k03_prenatal**: 2,121 missing (25.7%)
- **k04_prenatal_semanas**: 2,226 missing (27.0%) 
- **k06_peso_engravidar**: 2,108 missing (25.6%)
- **k07_peso_final**: 2,106 missing (25.6%)
- **k08_quilos**: 2,104 missing (25.5%)
- **k11_amamentou**: 2,113 missing (25.7%)
- **tempo_primeira_mamada_horas**: 2,379 missing (28.9%)
- **k15_recebeu**: 2,441 missing (29.6%)

## Treatment Strategy

✅ **Special codes** (99, 999.9) were replaced with **median of valid values**  
✅ **Categorical variables** were converted to **binary** (0/1)  
✅ **Missing values** were kept as **NaN** for later imputation  
✅ **Extreme values** in k08_quilos (≤-5kg) were treated as **invalid data**

The dataset is now ready for the next stage of the Feature Engineering pipeline.